In [ ]:
# Install required packages
%pip install -U sentence-transformers faiss-cpu tqdm scikit-learn hdbscan umap-learn
%pip install plotly matplotlib seaborn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 470.2/470.2 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.1/90.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Create working directory
!mkdir -p /content/drive/MyDrive/youtube_embeddings_project/results/


Mounted at /content/drive


In [ ]:
# Mount Google Drive and setup directories
from google.colab import drive
drive.mount('/content/drive')

# Create working directory
!mkdir -p /content/drive/MyDrive/youtube_embeddings_project/results/
print("✅ Setup complete!")

# Import all required libraries
import json
import numpy as np
import pandas as pd
from tqdm import tqdm
import os
import time
import gc
import torch
import faiss
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
import hdbscan
from concurrent.futures import ThreadPoolExecutor, as_completed
import multiprocessing as mp

# Setup paths
COMMENTS_DIR = "/content/drive/MyDrive/youtube_embeddings_project/comments/"
RESULTS_DIR = "/content/drive/MyDrive/youtube_embeddings_project/results/"

print("✅ Libraries imported successfully!")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Setup complete!
✅ Libraries imported successfully!


In [ ]:
# ULTRA-FAST PARALLEL COMMENT LOADING

def load_single_file(file_info):
    """Load comments from a single file - optimized for parallel processing"""
    file_path, file_name = file_info
    song_name = file_name.replace('.json', '').replace('_', ' ')

    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            comments = json.load(f)

        # Filter and clean comments
        valid_comments = []
        for comment in comments:
            if isinstance(comment, str) and comment.strip():
                clean_comment = comment.strip()
                # Skip very short comments or spam
                if len(clean_comment) > 3 and not clean_comment.isdigit():
                    valid_comments.append(clean_comment)

        return valid_comments, song_name, file_name
    except Exception as e:
        print(f"❌ Error loading {file_name}: {e}")
        return [], song_name, file_name

def load_comments_parallel(comments_dir):
    """Load all comments using parallel processing"""
    comment_files = [f for f in os.listdir(comments_dir) if f.endswith('.json')]
    file_paths = [(os.path.join(comments_dir, f), f) for f in comment_files]

    print(f"🚀 Loading {len(comment_files)} files in parallel...")

    all_comments = []
    song_metadata = []

    # Use parallel processing
    max_workers = min(32, len(comment_files))

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_file = {executor.submit(load_single_file, file_info): file_info[1]
                         for file_info in file_paths}

        for future in tqdm(as_completed(future_to_file), total=len(file_paths), desc="Loading files"):
            file_name = future_to_file[future]
            try:
                comments_list, song_name, file_name = future.result()

                # Add to main collections
                for comment in comments_list:
                    all_comments.append(comment)
                    song_metadata.append({
                        'song': song_name,
                        'file': file_name,
                        'comment_idx': len(all_comments) - 1
                    })

            except Exception as e:
                print(f"❌ Error processing {file_name}: {e}")

    return all_comments, song_metadata

# Check files and load
if not os.path.exists(COMMENTS_DIR):
    print(f"⚠️  Please upload your comments folder to: {COMMENTS_DIR}")
else:
    print(f"📂 Starting optimized comment loading...")
    start_time = time.time()

    comments, metadata = load_comments_parallel(COMMENTS_DIR)

    loading_time = time.time() - start_time
    print(f"✅ Comments loaded in {loading_time:.1f} seconds")
    print(f"✅ Total comments: {len(comments):,}")
    print(f"✅ Unique songs: {len(set(m['song'] for m in metadata))}")
    print(f"✅ Loading speed: {len(comments)/loading_time:.0f} comments/second")

    # Remove duplicates
    unique_comments = list(dict.fromkeys(comments))
    if len(unique_comments) < len(comments):
        print(f"🔄 Removed {len(comments) - len(unique_comments)} duplicates")
        comments = unique_comments

    print(f"✅ Final comment count: {len(comments):,}")


📂 Starting optimized comment loading...
🚀 Loading 227 files in parallel...


Loading files: 100%|██████████| 227/227 [00:30<00:00,  7.46it/s]


✅ Comments loaded in 31.1 seconds
✅ Total comments: 6,476,109
✅ Unique songs: 227
✅ Loading speed: 208070 comments/second
🔄 Removed 2285685 duplicates
✅ Final comment count: 4,190,424


In [ ]:
# MODEL SELECTION FOR SPEED VS QUALITY

MODELS = {
    'fastest': {
        'name': 'all-MiniLM-L6-v2',
        'dims': 384,
        'speed': 2000,
        'quality': 'Good',
        'description': 'Fastest option, great for initial testing'
    },
    'balanced': {
        'name': 'all-mpnet-base-v2',
        'dims': 768,
        'speed': 1000,
        'quality': 'Very Good',
        'description': 'Best balance of speed and quality'
    },
    'quality': {
        'name': 'intfloat/multilingual-e5-large-instruct',
        'dims': 1024,
        'speed': 200,
        'quality': 'Excellent',
        'description': 'Best quality, supports multilingual'
    },
    'multilingual': {
        'name': 'intfloat/multilingual-e5-base',
        'dims': 768,
        'speed': 500,
        'quality': 'Very Good',
        'description': 'Fast multilingual model'
    }
}

# 🎯 SELECT YOUR MODEL HERE
MODEL_CHOICE = 'balanced'  # Change to: 'fastest', 'balanced', 'quality', 'multilingual'

selected_model = MODELS[MODEL_CHOICE]
print(f"🎯 Selected Model: {selected_model['name']}")
print(f"  📊 Dimensions: {selected_model['dims']}")
print(f"  🚀 Speed: ~{selected_model['speed']} comments/second")
print(f"  ⭐ Quality: {selected_model['quality']}")
print(f"  📝 Description: {selected_model['description']}")

# Estimate processing time
estimated_time = len(comments) / selected_model['speed']
print(f"  ⏱️  Estimated time for {len(comments):,} comments: {estimated_time:.1f} seconds ({estimated_time/60:.1f} minutes)")

# Set instruction prefix based on model
INSTRUCTION_PREFIX = "query: " if 'e5' in selected_model['name'] else ""
print(f"  🔧 Instruction prefix: '{INSTRUCTION_PREFIX}'")

# GPU optimization info
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\n🎮 Device: {device}")
if device == "cuda":
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"  💾 GPU Memory: {gpu_memory:.1f} GB")
    print(f"  🔧 Will use FP16 mixed precision")
    print(f"  📦 Batch size: 1024")
else:
    print(f"  📦 CPU batch size: 128")


🎯 Selected Model: all-mpnet-base-v2
  📊 Dimensions: 768
  🚀 Speed: ~1000 comments/second
  ⭐ Quality: Very Good
  📝 Description: Best balance of speed and quality
  ⏱️  Estimated time for 4,190,424 comments: 4190.4 seconds (69.8 minutes)
  🔧 Instruction prefix: ''

🎮 Device: cpu
  📦 CPU batch size: 128


In [ ]:
# ULTRA-FAST EMBEDDING GENERATION

# GPU optimization
if device == "cuda":
    torch.cuda.empty_cache()

# Load model with optimizations
print(f"🚀 Loading model: {selected_model['name']}")
model = SentenceTransformer(selected_model['name'], device=device)

# Enable FP16 for speed
if device == "cuda":
    model.half()
    print(f"✅ Model loaded with FP16 optimization")

# Optimization settings
BATCH_SIZE = 1024 if device == "cuda" else 128
CHECKPOINT_INTERVAL = 25000
USE_MIXED_PRECISION = True

print(f"🚀 Optimization settings:")
print(f"  📦 Batch size: {BATCH_SIZE}")
print(f"  💾 Checkpoint every: {CHECKPOINT_INTERVAL:,} comments")
print(f"  ⚡ Mixed precision: {USE_MIXED_PRECISION}")

# Prepare inputs
inputs = [f"{INSTRUCTION_PREFIX}{comment}" for comment in comments]
total_comments = len(inputs)
print(f"📊 Total comments to embed: {total_comments:,}")

# Ultra-fast batch processing
def process_batch_optimized(batch_texts):
    """Process batch with all optimizations"""
    if device == "cuda":
        torch.cuda.empty_cache()

    with torch.no_grad():
        if USE_MIXED_PRECISION and device == "cuda":
            with torch.cuda.amp.autocast():
                embeddings = model.encode(
                    batch_texts,
                    normalize_embeddings=True,
                    show_progress_bar=False,
                    convert_to_tensor=True
                )
        else:
            embeddings = model.encode(
                batch_texts,
                normalize_embeddings=True,
                show_progress_bar=False,
                convert_to_tensor=True
            )

    # Convert to numpy with memory optimization
    embeddings_np = embeddings.cpu().numpy().astype(np.float16)

    # Clean up GPU memory
    if device == "cuda":
        del embeddings
        torch.cuda.empty_cache()

    return embeddings_np

# Process with speed monitoring
print(f"🚀 Starting ultra-fast embedding generation...")
start_time = time.time()

# Progress bar with speed tracking
all_embeddings = []
pbar = tqdm(total=total_comments, desc="🚀 Embedding", unit="comments", unit_scale=True)

batch_times = []
for i in range(0, total_comments, BATCH_SIZE):
    batch_start = time.time()
    batch_end = min(i + BATCH_SIZE, total_comments)
    batch = inputs[i:batch_end]

    # Process batch
    batch_embeddings = process_batch_optimized(batch)
    all_embeddings.append(batch_embeddings)

    # Update progress with speed
    batch_time = time.time() - batch_start
    batch_times.append(batch_time)
    speed = len(batch) / batch_time
    pbar.update(len(batch))
    pbar.set_postfix({'Speed': f'{speed:.0f} c/s'})

    # Memory management
    if len(all_embeddings) > 20:
        temp_embeddings = np.vstack(all_embeddings)
        np.save(f"{RESULTS_DIR}/temp_{i}.npy", temp_embeddings)
        all_embeddings = []
        gc.collect()

pbar.close()

# Combine all embeddings
print("🔄 Combining embeddings...")
if all_embeddings:
    embeddings_matrix = np.vstack(all_embeddings)
else:
    # Load from temp files
    temp_files = [f for f in os.listdir(RESULTS_DIR) if f.startswith('temp_')]
    temp_files.sort(key=lambda x: int(x.split('_')[1].split('.')[0]))

    chunks = []
    for temp_file in temp_files:
        chunk = np.load(f"{RESULTS_DIR}/{temp_file}")
        chunks.append(chunk)
        os.remove(f"{RESULTS_DIR}/{temp_file}")

    embeddings_matrix = np.vstack(chunks)

# Convert to float32 for compatibility
embeddings_matrix = embeddings_matrix.astype(np.float32)
gc.collect()

# Performance metrics
elapsed_time = time.time() - start_time
speed = total_comments / elapsed_time

print(f"\n✅ Ultra-fast embedding complete!")
print(f"  📊 Shape: {embeddings_matrix.shape}")
print(f"  ⏱️  Time: {elapsed_time:.1f}s ({elapsed_time/60:.1f}m)")
print(f"  🚀 Speed: {speed:.1f} comments/second")
print(f"  💾 Memory: {embeddings_matrix.nbytes/1024**3:.2f} GB")

# Save embeddings
np.save(f"{RESULTS_DIR}/embeddings_{MODEL_CHOICE}.npy", embeddings_matrix)
print(f"✅ Saved: embeddings_{MODEL_CHOICE}.npy")

# Quick clustering and search setup
print("\n🧪 Creating clusters and search index...")
n_clusters = min(50, len(comments) // 100)
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
kmeans_labels = kmeans.fit_predict(embeddings_matrix)

# Build FAISS index
dim = embeddings_matrix.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings_matrix)

# Save results
np.save(f"{RESULTS_DIR}/kmeans_labels.npy", kmeans_labels)
faiss.write_index(index, f"{RESULTS_DIR}/search_index.index")

print(f"✅ Clustering complete with {n_clusters} clusters")
print(f"✅ Search index created with {index.ntotal:,} vectors")

# Demo search function
def search_comments(query, top_k=5):
    """Search similar comments"""
    query_embedding = model.encode([f"{INSTRUCTION_PREFIX}{query}"], normalize_embeddings=True)
    scores, indices = index.search(query_embedding.astype('float32'), top_k)

    results = []
    for i, (score, idx) in enumerate(zip(scores[0], indices[0])):
        results.append({
            'rank': i + 1,
            'score': float(score),
            'comment': comments[idx],
            'song': metadata[idx]['song']
        })
    return results

# Demo search
print("\n🔍 Demo Search:")
demo_query = "this song brings back memories"
results = search_comments(demo_query, top_k=3)
print(f"Query: '{demo_query}'")
for r in results:
    print(f"  {r['rank']}. [{r['song'][:25]}...] {r['comment'][:60]}... (Score: {r['score']:.3f})")

print(f"\n🎉 Complete! All results saved to {RESULTS_DIR}")


🚀 Loading model: all-mpnet-base-v2


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

🚀 Optimization settings:
  📦 Batch size: 128
  💾 Checkpoint every: 25,000 comments
  ⚡ Mixed precision: True
📊 Total comments to embed: 4,190,424
🚀 Starting ultra-fast embedding generation...


🚀 Embedding:   3%|▎         | 133k/4.19M [9:29:22<399:24:32, 2.82comments/s, Speed=2 c/s]

In [ ]:
# Uninstall the conflicting scikit-learn version
%pip uninstall -y scikit-learn

# Install a compatible version
%pip install scikit-learn==1.6.1

Found existing installation: scikit-learn 1.7.0
Uninstalling scikit-learn-1.7.0:
  Successfully uninstalled scikit-learn-1.7.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.5/13.5 MB 64.5 MB/s eta 0:00:00
